# TripMe Part 6 — Provisional Gold Evaluation
Attach `tripme-part06-provisional-eval`, enable a T4 GPU, keep Internet on, and Run All. The 120 references are AI drafts, not human-approved gold labels.

In [ ]:
!pip install -q --no-cache-dir transformers==4.48.3 accelerate==1.3.0 peft==0.14.0 bitsandbytes==0.48.2 safetensors>=0.4
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
import torch, bitsandbytes as bnb
print('PyTorch:', torch.__version__, 'CUDA:', torch.version.cuda, 'bitsandbytes:', bnb.__version__)
assert torch.cuda.is_available(), 'GPU is not enabled'
assert torch.cuda.device_count() == 1

In [ ]:
from pathlib import Path
import json, re, shutil, statistics, time
MODEL_ID = 'Qwen/Qwen2.5-3B-Instruct'
MAX_NEW_TOKENS = 384
SEED = 42
matches = list(Path('/kaggle/input').rglob('provisional_gold_eval_120.jsonl'))
if not matches:
    raise FileNotFoundError('Attach tripme-part06-provisional-eval')
DATA_DIR = matches[0].parent
ADAPTER_DIR = DATA_DIR/'adapter'
OUTPUT_DIR = Path('/kaggle/working/tripme-part06-eval')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
rows = [json.loads(line) for line in matches[0].read_text(encoding='utf-8').splitlines() if line.strip()]
assert len(rows) == 120
assert (ADAPTER_DIR/'adapter_model.safetensors').is_file()
print('GPU:', torch.cuda.get_device_name(0), 'Rows:', len(rows), 'Data:', DATA_DIR)

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True)
tokenizer.pad_token = tokenizer.pad_token or tokenizer.eos_token
quant = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True,
)
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, quantization_config=quant, device_map={'': 0}, torch_dtype=torch.float16
)
model = PeftModel.from_pretrained(base_model, str(ADAPTER_DIR))
model.eval()
print('Adapter loaded:', ADAPTER_DIR)

In [ ]:
def generate_answer(row, use_adapter=True):
    messages = [
        {'role': 'system', 'content': row['system_prompt']},
        {'role': 'user', 'content': row['prompt']},
    ]
    inputs = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors='pt'
    ).to(model.device)
    start = time.time()
    context = model.disable_adapter() if not use_adapter else __import__('contextlib').nullcontext()
    with context, torch.no_grad():
        output = model.generate(
            inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False, repetition_penalty=1.05,
            pad_token_id=tokenizer.eos_token_id, eos_token_id=tokenizer.eos_token_id,
        )
    generated_ids = output[0][inputs.shape[-1]:]
    answer = tokenizer.decode(generated_ids, skip_special_tokens=True).strip()
    return answer, int(generated_ids.shape[-1]), round(time.time()-start, 3)

def normalized(text):
    return re.sub(r'[^\w\u0D80-\u0DFF\u0B80-\u0BFF]+', ' ', text.casefold()).strip()

def repetition_score(text):
    words = normalized(text).split()
    if len(words) < 6:
        return 0.0
    trigrams = [tuple(words[i:i+3]) for i in range(len(words)-2)]
    return round(1 - len(set(trigrams))/len(trigrams), 4)

def score_answer(row, answer, token_count):
    norm = normalized(answer)
    names = [normalized(name) for name in row['target_place_names']]
    name_coverage = sum(name in norm for name in names) / max(1, len(names))
    complete_ending = bool(answer) and answer.rstrip().endswith(('.', '!', '?', '।'))
    hit_token_limit = token_count >= MAX_NEW_TOKENS
    refusal_required = row['scenario'] in {'budget', 'uncertainty_current'}
    refusal_terms = {
        'si': ['තහවුරු කළ නොහැක', 'අනුමාන කළ නොහැක', 'මූලාශ්‍ර'],
        'singlish': ['confirm karanna ba', 'guess karanna ba', 'source'],
        'en': ['cannot confirm', 'cannot be confirmed', 'official source', 'recent source'],
    }
    refusal_ok = (not refusal_required) or any(term.casefold() in answer.casefold() for term in refusal_terms[row['language']])
    return {
        'nonempty': bool(answer), 'complete_ending': complete_ending,
        'hit_token_limit': hit_token_limit, 'place_name_coverage': round(name_coverage, 4),
        'refusal_compliance': refusal_ok, 'repetition_score': repetition_score(answer),
        'exact_reference_match': normalized(answer) == normalized(row['reference_answer']),
    }

In [ ]:
adapter_results = []
for index, row in enumerate(rows, 1):
    answer, tokens, seconds = generate_answer(row, use_adapter=True)
    adapter_results.append({
        'eval_id': row['eval_id'], 'language': row['language'], 'scenario': row['scenario'],
        'target_place_ids': row['target_place_ids'], 'target_place_names': row['target_place_names'],
        'prompt': row['prompt'], 'reference_answer': row['reference_answer'],
        'required_facts': row['required_facts'], 'forbidden_claims': row['forbidden_claims'],
        'generated': answer, 'generated_tokens': tokens, 'generation_seconds': seconds,
        'automatic_checks': score_answer(row, answer, tokens),
        'evaluation_status': 'provisional_ai_draft_not_human_gold',
    })
    if index % 10 == 0:
        print('Adapter completed:', index, '/', len(rows))

In [ ]:
# A 30-row deterministic base-model comparison limits Kaggle runtime.
base_rows = sorted(rows, key=lambda row: row['eval_id'])[:30]
base_results = []
for index, row in enumerate(base_rows, 1):
    answer, tokens, seconds = generate_answer(row, use_adapter=False)
    base_results.append({
        'eval_id': row['eval_id'], 'language': row['language'], 'scenario': row['scenario'],
        'prompt': row['prompt'], 'generated': answer, 'generated_tokens': tokens,
        'generation_seconds': seconds, 'automatic_checks': score_answer(row, answer, tokens),
    })
    if index % 10 == 0:
        print('Base completed:', index, '/', len(base_rows))

In [ ]:
def aggregate(results):
    checks = [row['automatic_checks'] for row in results]
    return {
        'rows': len(results),
        'nonempty_rate': sum(x['nonempty'] for x in checks)/len(checks),
        'complete_ending_rate': sum(x['complete_ending'] for x in checks)/len(checks),
        'token_limit_hit_rate': sum(x['hit_token_limit'] for x in checks)/len(checks),
        'mean_place_name_coverage': statistics.mean(x['place_name_coverage'] for x in checks),
        'refusal_compliance_rate': sum(x['refusal_compliance'] for x in checks)/len(checks),
        'mean_repetition_score': statistics.mean(x['repetition_score'] for x in checks),
        'exact_reference_match_rate': sum(x['exact_reference_match'] for x in checks)/len(checks),
        'mean_generation_seconds': statistics.mean(row['generation_seconds'] for row in results),
    }

summary = {
    'status': 'provisional_evaluation_complete_not_human_gold',
    'model_id': MODEL_ID, 'max_new_tokens': MAX_NEW_TOKENS,
    'adapter_all_120': aggregate(adapter_results),
    'base_comparison_30': aggregate(base_results),
    'adapter_by_language': {
        lang: aggregate([row for row in adapter_results if row['language'] == lang])
        for lang in ['si', 'singlish', 'en']
    },
    'warning': 'Automatic checks are diagnostic only; the references and results require human review.',
}
for filename, records in [('adapter_generations.jsonl', adapter_results), ('base_generations_30.jsonl', base_results)]:
    with (OUTPUT_DIR/filename).open('w', encoding='utf-8') as handle:
        for record in records:
            handle.write(json.dumps(record, ensure_ascii=False) + '\n')
(OUTPUT_DIR/'evaluation_summary.json').write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8')
archive = shutil.make_archive('/kaggle/working/tripme_part06_eval_output', 'zip', OUTPUT_DIR)
print(json.dumps(summary, ensure_ascii=False, indent=2))
print('Download:', archive)

In [ ]:
from IPython.display import HTML, display
zip_path = Path('/kaggle/working/tripme_part06_eval_output.zip')
assert zip_path.is_file()
print('Exists:', zip_path.exists(), 'Size MB:', round(zip_path.stat().st_size/1024/1024, 2))
display(HTML("<a href='files/tripme_part06_eval_output.zip' download>Download tripme_part06_eval_output.zip</a>"))